# High-Accuracy Simple 2D-CNN Facial Emotion Recognition (FER)
**Datasets Supported:** FER2013, AffectNet+, CK+, RAVDESS  
**Architecture:** Optimized 2D-CNN with CBAM (Paper 1 & Paper 3 Baseline)

In [1]:
import os
import sys
import glob
import json
import random
import yaml
import cv2
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torch.amp import autocast, GradScaler
import torchvision.transforms as T

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

CONFIG_PATH = "config.yaml"
DEFAULT_YAML = """
project:
  name: "Simple_2DCNN_FER"
  seed: 42
  device: "cuda"
  num_workers: 0
  mixed_precision: true
  output_dir: "./outputs"

datasets:
  use_fer2013: true
  use_affectnet: true
  use_ckplus: true
  use_ravdess: true
  paths:
    fer2013_dir: "C:\\Users\\Shahzaib Hussain\\Desktop\\Shahzaib Hussain\\FYP\\Datasets\\Facial Dataset\\FER2013"
    affectnet_dir: "C:\\Users\\Shahzaib Hussain\\Desktop\\Shahzaib Hussain\\FYP\\Datasets\\Facial Dataset\\AffectNet+"
    ckplus_dir: "C:\\Users\\Shahzaib Hussain\\Desktop\\Shahzaib Hussain\\FYP\\Datasets\\Facial Dataset\\CK+"
    ravdess_dir: "C:\\Users\\Shahzaib Hussain\\Desktop\\Shahzaib Hussain\\FYP\\Datasets\\Voice and Facial\\RAVDESS"

emotions:
  mode: "7_class"
  classes_7: ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

preprocessing:
  image_size: 224
  augmentation:
    horizontal_flip_prob: 0.5
    rotation_degrees: 15
    perspective_warp_prob: 0.3
    motion_blur_prob: 0.25
    color_jitter:
      brightness: 0.3
      contrast: 0.3
      saturation: 0.2
    random_erasing_prob: 0.25
    use_mixup_cutmix: true
    mixup_alpha: 0.2

training:
  epochs: 25
  batch_size: 64
  lr: 0.0003
  weight_decay: 0.005
  val_split_ratio: 0.15
  optimizer: "radam"
  label_smoothing: 0.1
  focal_gamma: 2.0
  early_stopping_patience: 6
"""

if not os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "w") as f:
        f.write(DEFAULT_YAML.strip())

with open(CONFIG_PATH, "r") as f:
    CFG = yaml.safe_load(f) or {}

# Supply robust fallbacks if loaded from a two-stage or custom config
CFG.setdefault("project", {})
CFG["project"].setdefault("num_workers", 0)
CFG.setdefault("training", {})
t_cfg = CFG["training"]
t_cfg.setdefault("batch_size", t_cfg.get("stage1_pretrain", {}).get("batch_size", 64))
t_cfg.setdefault("epochs", t_cfg.get("stage2_finetune", {}).get("epochs", 25))
t_cfg.setdefault("lr", t_cfg.get("stage1_pretrain", {}).get("lr", 0.0003))
t_cfg.setdefault("weight_decay", 0.005)
t_cfg.setdefault("val_split_ratio", 0.15)
t_cfg.setdefault("optimizer", "radam")
t_cfg.setdefault("label_smoothing", 0.1)
t_cfg.setdefault("focal_gamma", 2.0)
t_cfg.setdefault("early_stopping_patience", 6)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG["project"]["seed"])
os.makedirs(CFG["project"]["output_dir"], exist_ok=True)
DEVICE = torch.device(CFG["project"]["device"] if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Emotion Classes: {CFG['emotions']['classes_7']}")
print(f"DataLoader Workers: {CFG['project']['num_workers']} (Windows-safe)")


Device: cuda (NVIDIA GeForce RTX 5090)
Emotion Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
DataLoader Workers: 0 (Windows-safe)


## 1. Unified Multi-Dataset Ingestion Engine
Seamlessly loads and harmonizes **FER2013**, **CK+**, **AffectNet+**, and **RAVDESS** into a single balanced dataset.

In [2]:
TARGET_CLASSES = CFG["emotions"]["classes_7"]
CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {idx: cls_name for cls_name, idx in CLASS_TO_IDX.items()}

class UnifiedFERDataset(Dataset):
    """
    Harmonized Facial Emotion Recognition Dataset.
    Supports:
      - FER2013: standard train/test folders
      - CK+: ckextended.csv pixel strings
      - AffectNet+: cached balanced manifest or JSON annotations
      - RAVDESS: cached facial crops manifest
    """
    def __init__(self, ds_type, root_dir, split="train", val_split_ratio=0.15, seed=42, transform=None):
        self.ds_type = ds_type.lower()
        self.root_dir = str(root_dir)
        self.split = split
        self.val_split_ratio = val_split_ratio
        self.seed = seed
        self.transform = transform
        self.samples = []
        self._load()

    def _load(self):
        valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        discovered = []

        # 1. FER2013 (train/test folders)
        if "fer2013" in self.ds_type:
            s_folder = "train" if self.split == "train" else "test"
            s_dir = os.path.join(self.root_dir, s_folder)
            if os.path.exists(s_dir):
                for cname in sorted(os.listdir(s_dir)):
                    canonical = cname.lower()
                    if canonical in CLASS_TO_IDX:
                        cdir = os.path.join(s_dir, cname)
                        if os.path.isdir(cdir):
                            for f in os.listdir(cdir):
                                if f.lower().endswith(valid_exts):
                                    discovered.append((os.path.join(cdir, f), CLASS_TO_IDX[canonical]))
                self.samples = discovered

        # 2. CK+ (ckextended.csv pixel strings)
        elif "ck" in self.ds_type:
            csv_path = os.path.join(self.root_dir, "ckextended.csv")
            ck_map = {0: "neutral", 1: "angry", 3: "disgust", 4: "fear", 5: "happy", 6: "sad", 7: "surprise"}
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                for _, row in df.iterrows():
                    emo = int(row["emotion"])
                    if emo in ck_map:
                        cname = ck_map[emo]
                        px = np.fromstring(str(row["pixels"]), sep=" ", dtype=np.uint8).reshape(48, 48)
                        discovered.append((px, CLASS_TO_IDX[cname]))
                rng = random.Random(self.seed)
                rng.shuffle(discovered)
                val_sz = int(len(discovered) * self.val_split_ratio)
                self.samples = discovered[val_sz:] if self.split == "train" else discovered[:val_sz]

        # 3. AffectNet+ (cache manifest or subfolders)
        elif "affectnet" in self.ds_type:
            cache_candidates = [
                "cache/affectnet_balanced_manifest.json",
                "./cache/affectnet_balanced_manifest.json",
                "../FER 95 +/cache/affectnet_balanced_manifest.json",
                "../New Notebook2/cache_gpu/affectnet_balanced_manifest.json"
            ]
            for cp in cache_candidates:
                if os.path.exists(cp):
                    with open(cp, "r", encoding="utf-8") as f:
                        items = json.load(f)
                    for it in items:
                        it_split = it.get("split", "train")
                        if (self.split == "train" and it_split == "train") or (self.split == "val" and it_split != "train"):
                            discovered.append((it["data"], it["label"]))
                    break
            self.samples = discovered

        # 4. RAVDESS (cache manifest)
        elif "ravdess" in self.ds_type:
            cache_candidates = [
                "cache/ravdess_balanced_manifest.json",
                "./cache/ravdess_balanced_manifest.json",
                "../FER 95 +/cache/ravdess_balanced_manifest.json",
                "../New Notebook2/cache_gpu/ravdess_balanced_manifest.json"
            ]
            for cp in cache_candidates:
                if os.path.exists(cp):
                    with open(cp, "r", encoding="utf-8") as f:
                        items = json.load(f)
                    for it in items:
                        it_split = it.get("split", "train")
                        if (self.split == "train" and it_split == "train") or (self.split == "val" and it_split != "train"):
                            discovered.append((it["data"], it["label"]))
                    break
            self.samples = discovered

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item, label = self.samples[idx]
        if isinstance(item, np.ndarray):
            img = Image.fromarray(item).convert("RGB")
        elif isinstance(item, str):
            with Image.open(item) as im:
                img = im.convert("RGB")
        else:
            img = item.convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

def get_transforms(cfg):
    sz = cfg["preprocessing"]["image_size"]
    aug = cfg["preprocessing"]["augmentation"]
    train_tf = T.Compose([
        T.Resize((sz + 16, sz + 16)),
        T.RandomCrop((sz, sz)),
        T.RandomHorizontalFlip(p=aug["horizontal_flip_prob"]),
        T.RandomRotation(degrees=aug["rotation_degrees"]),
        T.RandomPerspective(distortion_scale=0.2, p=aug["perspective_warp_prob"]),
        T.ColorJitter(brightness=aug["color_jitter"]["brightness"], contrast=aug["color_jitter"]["contrast"], saturation=aug["color_jitter"]["saturation"]),
        T.RandomApply([T.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0))], p=aug["motion_blur_prob"]),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        T.RandomErasing(p=aug["random_erasing_prob"], scale=(0.02, 0.15))
    ])
    val_tf = T.Compose([
        T.Resize((sz, sz)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_tf, val_tf

def build_weighted_dataloader(datasets_list, batch_size, num_workers=0, is_train=True):
    valid_ds = [d for d in datasets_list if len(d) > 0]
    if not valid_ds:
        return None
    combined = ConcatDataset(valid_ds)
    if not is_train:
        return DataLoader(combined, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    all_labels = [s[1] for d in valid_ds for s in d.samples]
    counts = Counter(all_labels)
    total = len(all_labels)
    cls_weights = {c: total / (len(CLASS_TO_IDX) * cnt) for c, cnt in counts.items()}
    sample_weights = [cls_weights[l] for l in all_labels]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    return DataLoader(combined, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=True, drop_last=True)


## 2. Simple High-Accuracy 2D-CNN Architecture with CBAM (Paper 1 & Paper 3)
Implements a lightweight yet powerful 4-stage convolutional neural network equipped with a Convolutional Block Attention Module (**CBAM**) and adaptive pooling.

In [3]:
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, max(1, channels // reduction), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(1, channels // reduction), channels, 1, bias=False)
        )
        self.spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        ca = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        x = x * ca
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa = self.sigmoid(self.spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * sa

class Simple2DCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.35):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),  # 112x112
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), # 56x56
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2), # 28x28
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), # 28x28
            CBAMBlock(512),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(num_classes=7):
    return Simple2DCNN(num_classes=num_classes, dropout=0.35).to(DEVICE)

# Quick sanity check
_test_m = build_model(len(TARGET_CLASSES))
_dummy = torch.randn(2, 3, 224, 224, device=DEVICE)
with torch.no_grad():
    _out = _test_m(_dummy)
print(f"Simple 2D-CNN Model Initialized! Output shape: {_out.shape}")


Simple 2D-CNN Model Initialized! Output shape: torch.Size([2, 7])


## 3. Loss, Optimizer & Unified Training Engine
Uses **Smoothed Focal Loss** for class imbalance mitigation and **MixUp** data augmentation for high generalization.

In [4]:
class SmoothedFocalLoss(nn.Module):
    def __init__(self, num_classes, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, logits, targets):
        conf = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.num_classes - 1)
        with torch.no_grad():
            true_dist = torch.full_like(logits, smooth_val)
            true_dist.scatter_(1, targets.unsqueeze(1), conf)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        focal_weight = (1.0 - probs) ** self.gamma
        return torch.mean(-torch.sum(true_dist * focal_weight * log_probs, dim=-1))

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx, :], y, y[idx], lam

def train_one_epoch(model, loader, optimizer, criterion, scaler, cfg):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = cfg["project"]["mixed_precision"] and DEVICE.type == "cuda"
    use_mixup = cfg["preprocessing"]["augmentation"]["use_mixup_cutmix"]
    alpha = cfg["preprocessing"]["augmentation"]["mixup_alpha"]

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        do_mix = use_mixup and (random.random() < 0.4)
        if do_mix:
            images, ya, yb, lam = mixup_data(images, labels, alpha)
        with autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss = (lam * criterion(outputs, ya) + (1 - lam) * criterion(outputs, yb)) if do_mix else criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{(correct/total)*100:.2f}%"})
    return total_loss / total, (correct / total) * 100.0

@torch.no_grad()
def evaluate(model, loader, criterion, cfg):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds) * 100.0
    f1 = f1_score(all_labels, all_preds, average="macro") * 100.0
    return total_loss / len(all_labels), acc, f1

def run_training(model, train_loader, val_loader, cfg):
    if train_loader is None or val_loader is None:
        print("Error: Train or Val DataLoader is empty!")
        return model

    t_cfg = cfg.get("training", {})
    epochs = t_cfg.get("epochs", t_cfg.get("stage2_finetune", {}).get("epochs", 25))
    lr = t_cfg.get("lr", t_cfg.get("stage1_pretrain", {}).get("lr", 0.0003))
    wd = t_cfg.get("weight_decay", 0.005)

    optimizer = torch.optim.RAdam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)
    criterion = SmoothedFocalLoss(len(TARGET_CLASSES), gamma=t_cfg.get("focal_gamma", 2.0), smoothing=t_cfg.get("label_smoothing", 0.1))
    scaler = GradScaler('cuda', enabled=(cfg["project"]["mixed_precision"] and DEVICE.type == "cuda"))

    best_acc, wait = 0.0, 0
    patience = t_cfg.get("early_stopping_patience", 6)
    save_path = os.path.join(cfg["project"]["output_dir"], "best_simple_2dcnn_model.pth")

    print("\n" + "="*60)
    print(f"STARTING SIMPLE 2D-CNN TRAINING ({epochs} Epochs)")
    print("="*60)

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, cfg)
        va_loss, va_acc, va_f1 = evaluate(model, val_loader, criterion, cfg)
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        is_best = va_acc > best_acc
        if is_best:
            best_acc = va_acc
            wait = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_acc": best_acc, "classes": TARGET_CLASSES}, save_path)
            best_tag = f"--> Best Model Saved ({best_acc:.2f}%)"
        else:
            wait += 1
            best_tag = f"(patience {wait}/{patience})"

        print(f"Epoch [{epoch:02d}/{epochs:02d}] LR: {current_lr:.6f} | Train Acc: {tr_acc:.2f}% | Val Acc: {va_acc:.2f}% | F1: {va_f1:.2f}% | {best_tag}")

        if wait >= patience:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

    if os.path.exists(save_path):
        ckpt = torch.load(save_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded best model with Validation Accuracy: {ckpt['val_acc']:.2f}%")
    return model


## 4. Initialize & Run Training Pipeline
Aggregates all enabled datasets into unified train & validation data loaders and begins training.

In [ ]:
train_tf, val_tf = get_transforms(CFG)
paths = CFG["datasets"]["paths"]
val_ratio = CFG["training"]["val_split_ratio"]
seed = CFG["project"]["seed"]

train_datasets = []
val_datasets = []

# 1. FER2013
if CFG["datasets"].get("use_fer2013", True):
    d_tr = UnifiedFERDataset("fer2013", paths["fer2013_dir"], "train", val_ratio, seed, train_tf)
    d_va = UnifiedFERDataset("fer2013", paths["fer2013_dir"], "val", val_ratio, seed, val_tf)
    train_datasets.append(d_tr)
    val_datasets.append(d_va)
    print(f"[FER2013] Loaded: {len(d_tr):,} train, {len(d_va):,} val")

# 2. AffectNet+
if CFG["datasets"].get("use_affectnet", True):
    d_tr = UnifiedFERDataset("affectnet", paths["affectnet_dir"], "train", val_ratio, seed, train_tf)
    d_va = UnifiedFERDataset("affectnet", paths["affectnet_dir"], "val", val_ratio, seed, val_tf)
    train_datasets.append(d_tr)
    val_datasets.append(d_va)
    print(f"[AffectNet+] Loaded: {len(d_tr):,} train, {len(d_va):,} val")

# 3. CK+
if CFG["datasets"].get("use_ckplus", True):
    d_tr = UnifiedFERDataset("ckplus", paths["ckplus_dir"], "train", val_ratio, seed, train_tf)
    d_va = UnifiedFERDataset("ckplus", paths["ckplus_dir"], "val", val_ratio, seed, val_tf)
    train_datasets.append(d_tr)
    val_datasets.append(d_va)
    print(f"[CK+] Loaded: {len(d_tr):,} train, {len(d_va):,} val")

# 4. RAVDESS
if CFG["datasets"].get("use_ravdess", True):
    d_tr = UnifiedFERDataset("ravdess", paths["ravdess_dir"], "train", val_ratio, seed, train_tf)
    d_va = UnifiedFERDataset("ravdess", paths["ravdess_dir"], "val", val_ratio, seed, val_tf)
    train_datasets.append(d_tr)
    val_datasets.append(d_va)
    print(f"[RAVDESS] Loaded: {len(d_tr):,} train, {len(d_va):,} val")

total_tr = sum(len(d) for d in train_datasets)
total_va = sum(len(d) for d in val_datasets)

print("\n" + "="*60)
print("DATASET STATISTICS (UNIFIED FER)")
print("="*60)
print(f"Total Training Samples:   {total_tr:,}")
print(f"Total Validation Samples: {total_va:,}")
print("="*60 + "\n")

batch_size = CFG.get("training", {}).get("batch_size", CFG.get("training", {}).get("stage1_pretrain", {}).get("batch_size", 64))
workers = int(CFG.get("project", {}).get("num_workers", 0))
if sys.platform == 'win32':
    workers = 0

train_loader = build_weighted_dataloader(train_datasets, batch_size, workers, is_train=True)
val_loader = build_weighted_dataloader(val_datasets, batch_size, workers, is_train=False)

model = build_model(len(TARGET_CLASSES))
model = run_training(model, train_loader, val_loader, CFG)
print("\nSimple 2D-CNN Training Pipeline Completed Successfully!")


[FER2013] Loaded: 28,709 train, 7,178 val
[AffectNet+] Loaded: 24,500 train, 3,500 val
[CK+] Loaded: 761 train, 134 val
[RAVDESS] Loaded: 9,288 train, 3,168 val

DATASET STATISTICS (UNIFIED FER)
Total Training Samples:   63,258
Total Validation Samples: 13,980


STARTING SIMPLE 2D-CNN TRAINING (25 Epochs)


Training:   0%|          | 0/988 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch [01/25] LR: 0.000299 | Train Acc: 19.50% | Val Acc: 18.33% | F1: 13.43% | --> Best Model Saved (18.33%)


Training:   0%|          | 0/988 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/219 [00:00<?, ?it/s]

... (11 outputs left out)

Evaluating:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch [06/25] LR: 0.000260 | Train Acc: 19.69% | Val Acc: 18.47% | F1: 8.60% | --> Best Model Saved (18.47%)


Training:   0%|          | 0/988 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/219 [00:00<?, ?it/s]

## 5. Evaluation & Visualization
Generates per-class classification metrics and plots the Confusion Matrix.

In [ ]:
print("\n" + "="*60)
print("FINAL EVALUATION ON VALIDATION DATA")
print("="*60)

if val_loader is not None:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Final Evaluation"):
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(labels.numpy())

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=TARGET_CLASSES, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
    plt.title("Confusion Matrix - Simple 2D-CNN with CBAM")
    plt.ylabel("True Emotion")
    plt.xlabel("Predicted Emotion")
    plt.tight_layout()
    cm_save = os.path.join(CFG["project"]["output_dir"], "confusion_matrix_simple2dcnn.png")
    plt.savefig(cm_save, dpi=150)
    plt.show()
    print(f"Confusion matrix saved to: {cm_save}")

    # Per-class accuracy
    print("\nPer-Class Accuracy:")
    for i, cls_name in enumerate(TARGET_CLASSES):
        mask = np.array(all_labels) == i
        if mask.sum() > 0:
            cls_acc = (np.array(all_preds)[mask] == i).mean() * 100
            print(f"  {cls_name:>10s}: {cls_acc:.2f}%")
else:
    print("No validation data available for final evaluation.")
